In [ ]:
#============================================
# Celda 1 — Carga parquet raw
#============================================
import pandas as pd, numpy as np, os
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent
os.chdir(ROOT)
print(f"✅ ROOT: {ROOT}")

PATH_RAW = ROOT / 'output/idealista/01-raw/raw_idealista_anuncios.parquet'

if not PATH_RAW.exists():
    raise FileNotFoundError(f"❌ No encontrado: {PATH_RAW}\n   Ejecuta primero 01_idealista.ipynb")

df = pd.read_parquet(PATH_RAW)
print(f"✅ Cargado: {df.shape[0]} anuncios × {df.shape[1]} columnas")
print(f"   Cities : {sorted(df['city'].unique())}")
print(f"   Ops    : {sorted(df['operation'].unique())}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")
df.head()

In [ ]:
#============================================
# Celda 2 — Limpieza general
#============================================

# Tipos correctos
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], errors='coerce')

cols_num = ['price','size_m2','price_m2','rooms','bathrooms','latitude','longitude']
for c in cols_num:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

cols_bool = ['exterior','newDevelopment']
for c in cols_bool:
    if c in df.columns:
        df[c] = df[c].astype('boolean')

# Deduplicar por código de anuncio
antes = len(df)
df = df.drop_duplicates(subset=['propertyCode'])
print(f"   Duplicados eliminados: {antes - len(df)}")

# Limpiar strings
for c in ['city','operation','district','municipality','neighborhood','province','status']:
    if c in df.columns:
        df[c] = df[c].str.strip().str.lower()

print(f"✅ df limpio: {df.shape}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")
df.head()

In [ ]:
#============================================
# Celda 3 — Imputación de nulos
#============================================

# floor: convertir TODO a string limpio para evitar tipo mixto
# (la columna tiene 'bj', 'ático', '1', '2'... y nulos)
df['floor'] = df['floor'].fillna('desconocido').astype(str).str.strip()

# exterior: nulo → False, luego bool puro
df['exterior'] = df['exterior'].fillna(False).astype(bool)

# district/neighborhood: nulo → 'desconocido'
df['district']     = df['district'].fillna('desconocido').astype(str)
df['neighborhood'] = df['neighborhood'].fillna('desconocido').astype(str)

# Outliers price_m2 — eliminar extremos (<1% y >99%)
for op in df['operation'].unique():
    mask = df['operation'] == op
    q01  = df.loc[mask, 'price_m2'].quantile(0.01)
    q99  = df.loc[mask, 'price_m2'].quantile(0.99)
    antes = mask.sum()
    df = df[~(mask & ((df['price_m2'] < q01) | (df['price_m2'] > q99)))]
    print(f"   [{op}] outliers eliminados: {antes - (df['operation']==op).sum()}")

# Verificar que no quedan tipos mixtos
tipos_problematicos = df.select_dtypes(include='object').apply(
    lambda col: col.dropna().map(type).nunique() > 1
)
if tipos_problematicos.any():
    print(f"⚠️  Columnas con tipos mixtos: {list(tipos_problematicos[tipos_problematicos].index)}")
else:
    print("✅ Sin tipos mixtos — parquet serializable")

print(f"\n✅ Tras imputación y outliers: {df.shape}")
print(f"   Nulos %: {round(df.isnull().sum().sum()/df.size*100,1)}%")

In [ ]:
#============================================
# Celda 4 — Features derivadas
#============================================

# Ratio precio/habitación
df['price_per_room'] = (df['price'] / df['rooms'].replace(0, np.nan)).round(2)

# Categoría de tamaño
bins_size   = [0, 50, 80, 120, 200, 9999]
labels_size = ['micro','pequeño','mediano','grande','lujo']
df['size_cat'] = pd.cut(df['size_m2'], bins=bins_size, labels=labels_size)

# Categoría de precio/m2 por operación
for op, bins, labels in [
    ('sale',  [0,2000,4000,6000,9000,99999], ['muy_bajo','bajo','medio','alto','premium']),
    ('rent',  [0,8,13,18,25,999],            ['muy_bajo','bajo','medio','alto','premium']),
]:
    mask = df['operation'] == op
    df.loc[mask, 'precio_cat'] = pd.cut(
        df.loc[mask, 'price_m2'], bins=bins, labels=labels
    )

print(f"✅ Features añadidas: price_per_room, size_cat, precio_cat")
df[['city','operation','price','size_m2','price_m2','size_cat','precio_cat']].head(10)

In [ ]:
#============================================
# Celda 5 — Resumen calidad final
#============================================
resumen = pd.DataFrame([{
    'tabla'    : 'idealista_anuncios',
    'filas'    : len(df),
    'columnas' : len(df.columns),
    'cities'   : df['city'].nunique(),
    'ops'      : df['operation'].nunique(),
    'snapshot' : df['snapshot_date'].max().date(),
    'nulos_%'  : round(df.isnull().sum().sum() / df.size * 100, 1)
}])
print(resumen.to_string(index=False))

print(f"\nPrice/m2 por operación:")
print(df.groupby('operation')['price_m2'].agg(['mean','median','min','max']).round(2))

In [ ]:
#============================================
# Celda 6 — Guardar parquet clean
#============================================
import pyarrow as pa

# ── Sanitizar TODAS las columnas object con tipos mixtos antes de guardar ──
for col in df.select_dtypes(include='object').columns:
    tipos = df[col].dropna().map(type).unique()
    if len(tipos) > 1:
        print(f"   ⚠️  Tipo mixto en '{col}': {tipos} → convirtiendo a str")
        df[col] = df[col].astype(str)

# ── Sanitizar columnas pd.Categorical (pueden tener problemas con pyarrow) ──
for col in df.select_dtypes(include='category').columns:
    df[col] = df[col].astype(str)

os.makedirs(str(ROOT / 'output/idealista/02-clean'), exist_ok=True)
ruta_clean = str(ROOT / 'output/idealista/02-clean/clean_idealista_anuncios.parquet')
df.to_parquet(ruta_clean, index=False)

print(f"✅ Guardado: output/idealista/02-clean/clean_idealista_anuncios.parquet")
print(f"   Filas   : {len(df)}")
print(f"   Columnas: {len(df.columns)}")
print(f"\n🎯 Pipeline 02_idealista completado — listo para análisis")